## 4.4 自动微分 - 在DL中的使用场景

#### 1. Autograd 在DL中的核心目标：
* 用 Autograd 自动算出每个参数的梯度，然后用优化器更新参数，让 Loss 下降。

#### 2. 训练闭环（Training Loop）的标准流程：

以后写任何 PyTorch 模型训练代码，都离不开这 6 步：
1. 准备数据（Data）
2. 前向传播（Forward）：算预测 y_pred
3. 计算损失（Loss）：预测和真实的差距
4. 反向传播（Backward）：loss.backward() 自动算梯度
5. 参数更新（Update）：optimizer.step() 用梯度更新参数
6. 梯度清零（Zero Grad）：optimizer.zero_grad() 防止梯度累加

#### 3. 真实案例：用一个最小神经网络跑通整个流程 🧪

深度学习训练闭环的最小原型，我们用 nn.Linear 做一个线性回归

##### 3.1 准备数据：y = 2x + 1（我们让模型学出来）

In [1]:
import torch
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])  # shape: (4,1)
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])  # shape: (4,1)

##### 3.2 定义模型：1个线性层（自动包含 W 和 b）
* 1个输入层特征
* 1个输出层特征

In [2]:
import torch.nn as nn # nn是neural network的缩写，包含了构建神经网络的各种组件和工具
model = nn.Linear(in_features=1, out_features=1) # 定义一个线性层，输入特征数为1，输出特征数为1

##### 3.3 定义损失函数为 MSE

In [3]:
criterion = nn.MSELoss() # 定义损失函数为均方误差（Mean Squared Error）

##### 3.4 优化器：SGD（随机梯度下降）

梯度下降的种类：
1. 全梯度下降：
    * 使用所有样本
    * 计算完整的损失函数梯度
    * 更新一次参数
2. **随机梯度下降（Stochastic Gradient Descent，SGD）**
    * 随机抽取 1 个样本
    * 立刻计算梯度
    * 立刻更新参数
3. 小批量梯度下降（Mini-batch Gradient Descent）
    * 把数据分成很多小批次（batch）
    * 每次用一个 batch 计算梯度
    * 更新一次参数
4. 随机平均梯度下降（SAG，Stochastic Average Gradient）
    * 每次随机选择一个样本进行梯度计算，保存每个样本的梯度到列表
    * 后续每次计算都使用列表中“所有历史梯度的平均值”来更新参数

In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

##### 3.5 训练循环

In [6]:
epochs = 30
for epoch in range(epochs):
    # 1. 前向传播
    y_pred = model(x)
    # 2. 计算损失
    loss = criterion(y_pred, y)
    print("Before training epoch: {} loss: {} w: {} b: {}".format(epoch, loss.item(), model.weight.item(), model.bias.item()))
    # 3. 反向传播
    loss.backward()
    # 4. 更新权重
    optimizer.step()
    print("After training epoch: {} loss: {} w: {} b: {}".format(epoch, loss.item(), model.weight.item(), model.bias.item()))
    # 5. 清零梯度
    optimizer.zero_grad()

Before training epoch: 0 loss: 0.0166921429336071 w: 2.107511043548584 b: 0.6838525533676147
After training epoch: 0 loss: 0.0166921429336071 w: 2.104318141937256 b: 0.6933265328407288
Before training epoch: 1 loss: 0.015707677230238914 w: 2.104318141937256 b: 0.6933265328407288
After training epoch: 1 loss: 0.015707677230238914 w: 2.10117769241333 b: 0.7025021314620972
Before training epoch: 2 loss: 0.014781174249947071 w: 2.10117769241333 b: 0.7025021314620972
After training epoch: 2 loss: 0.014781174249947071 w: 2.0981600284576416 b: 0.7114128470420837
Before training epoch: 3 loss: 0.013909334316849709 w: 2.0981600284576416 b: 0.7114128470420837
After training epoch: 3 loss: 0.013909334316849709 w: 2.0952136516571045 b: 0.7200502753257751
Before training epoch: 4 loss: 0.013088936917483807 w: 2.0952136516571045 b: 0.7200502753257751
After training epoch: 4 loss: 0.013088936917483807 w: 2.0923681259155273 b: 0.7284334301948547
Before training epoch: 5 loss: 0.012316963635385036 w: 2

#### 4. 把流程拆开讲透：每一步 Autograd 在幕后干了什么？

##### 4.1 Forward：y_pred = model(x) 做了什么？

nn.Linear 内部有两个可学习参数：
1. weight (W)
2. bias (b)

Autograd 在这一步做的事情：
1. 记录运算过程
2. 构建计算图（动态图）

##### 4.2  Loss：loss = criterion(y_pred, y) 做了什么？

MSELoss 计算均方误差

Autograd 在这一步：把 “loss 和 y_pred 的关系” 也加进计算图

最终形成： `x -> Linear(W,b) -> y_pred -> MSE(y_pred,y) -> loss`

##### 4.3 Backward：loss.backward() 到底发生了什么？

这一句是 Autograd 的核心入口。

Autograd 会做：
1. 从 loss 节点开始，向后遍历计算图
2. 使用链式法则计算梯度
3. 把每个参数的梯度存进 .grad

例如：
* model.weight.grad
* model.bias.grad

##### 4.4 Step：optimizer.step() 到底更新了什么？
SGD 的更新规则（概念上）:
* w新 = w旧 - 学习率*梯度
* b新 = b旧 - 学习率*梯度

📌 关键点：
* optimizer 读取 .grad
* 按自己的算法更新参数
* 这一步一般在 no_grad 环境下执行（内部已经处理）

##### 4.5 Zero Grad：为什么先 optimizer.zero_grad()？
因为 .grad 是累加的：
如果不清零，会导致：
* 梯度越来越大
* 更新方向混乱
* loss 可能反而上升

这一步不属于 Autograd 运算，但它保证 Autograd 的结果可用

#### 5. Autograd 在 DL 中的典型真实使用场景（经常遇到）

##### 5.1 标准监督学习训练（最常见）
* 分类：CrossEntropyLoss
* 回归：MSELoss
流程就是我们上面那套闭环。

##### 5.2 冻结部分网络（Transfer Learning 常用）🧊
* 比如你只训练最后一层：
* Autograd 表现：
    * requires_grad=False 的参数不会计算梯度，也不会更新

##### 5.3 验证/推理阶段关闭梯度（提升速度 + 省显存）

##### 5.4 只想训练一部分计算路径（detach 的真实场景）
例如把模型输出当 target